In [1]:
import pandas as pd
import os
import warnings
from pandasql import sqldf
from datetime import datetime, timedelta
import glob
warnings.filterwarnings("ignore")
import openpyxl

### Nielsen

In [2]:
dir = os.getcwd()
# Important!! Make sure the file exist and refreshed first
xls = pd.ExcelFile(f'{dir}/../Data Source/Nielsen/Nielsen O+O May26_230626.xlsx')
sheet_names = ['MY Nielsen Skincare', 'MY Nielsen Cosmetic', 'MY Nielsen Mass Medic']
dfs = {}
# Read each sheet into a DataFrame and store it in the dictionary
for sheet_name in sheet_names:
    dfs[sheet_name] = pd.read_excel(xls, sheet_name=sheet_name)
    
print(dir)

c:\Users\balatarsini_avinitya\Downloads\CPD & LDB O+O - May\loreal-report-automation (2)\O+O


In [3]:
for df in dfs:
    print(df)

MY Nielsen Skincare
MY Nielsen Cosmetic
MY Nielsen Mass Medic


In [4]:
dfs['MY Nielsen Skincare'][(dfs['MY Nielsen Skincare']['Periods'].str.contains('23')) &
                           dfs['MY Nielsen Skincare']['BRAND'].isna() 
                        #    dfs['MY Nielsen Skincare']['BRAND'] == ''
                           ][['Sales Value', 'Sales Units']].sum().apply(lambda x: f"{x:,.0f}")

Sales Value    1,988,025,342
Sales Units      106,250,335
dtype: str

In [5]:
def month_to_number(month):
    months = {'Jan': 1, 'Feb': 2, 'Mar': 3, 'Apr': 4, 'May': 5, 'Jun': 6, 'Jul': 7, 'Aug': 8, 'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dec': 12}
    return months.get(month, month)

In [6]:
# Rename Nielsen Platforms
def map_platform(row):
    if row['Markets'] in ['Pen Malaysia + EM Modern Trade', 'Total Malaysia Key Account']:
        return 'Nielsen'
    elif row['Markets'] == 'PM MT Drugstore/Pharmacy':
        return 'NS Drugstore'
    elif row['Markets'] == 'PM MT Hmkt':
        return 'NS Hypermarket'
    elif row['Markets'] == 'PM MT Smkt':
        return 'NS Supermarket'
    elif row['Markets'] == 'PM MT Mini':
        return 'NS Minimarket'
    elif row['Markets'] == 'PM Trad Trade':
        return 'NS Trad. Trade'
    elif row['Markets'] == 'East Malaysia Modern Trade':
        return 'EM Modern Trade'
    else:
        return 'Others'

In [7]:
# Important!! Brand Mapping - Any new Brands need to be added here - LDB Brands - Make sure consistent for all O+O Scripts
## mass_medic = ['ACNE AID', 'ACNES', 'AVEENO', 'BENZAC', 'BIAFINE', 'BIO-OIL', 'CARMEX', 'CERAVE', 'CETAPHIL', 'CUREL', 'DERMATIX', 'DERMAVEEN', 'DR.G', 'DR.YU', 'EGO', 'LINOLA', 'MUSTELA', 'NEUTROGENA', 'PANOXYL', 'PHYSIOGEL', 'SEBAMED', 'TOPICREM', 'VANICREAM', 'WIS', 'XHEKPON', 'QV']
# mass_medic = [
#     'ACNE AID', 'ACNES', 'AVEENO', 'BALNEUM', 'BENZAC', 'BIO-OIL', 'CARMEX', 'CERAVE', 'CETAPHIL', 'CUREL',
#     'DERMATIX', 'DERMAVEEN', 'DIFFERIN', 'DR.G', 'DR.YU', 'EGO', 'EUBOS', 'LACTACYD', 'LINOLA', 'MUSTELA',
#     'NEUTROGENA', 'PANOXYL', 'PHYSIOGEL', 'SEBAMED', 'TOPICREM', 'VANICREAM', 'WIS', 'XHEKPON', 'FIRST AID BEAUTY',
#     'AQUAPHOR', 'NOBACTER', 'LUBRIDERM', 'NEOSPORIN', 'DARROW', 'DEXERYL', 'ALERGIBON', 'ALPHYGIENE', 'BABIGOZ',
#     'CANDERMYL', 'GALDERMA', 'GALDERMA OTHER', 'HELIOBLOC', 'HYDRODERM OMEGA', 'IOCON', 'IONIL', 'MACROLANE',
#     'MICROBAN', 'MICROSUN', 'NESTLE', 'NUTRASPA', 'OBSERVANCE', 'PHYGIENE', 'R-GEN', 'SENTIAL', 'ACHE', 'ACNAID',
#     'ACNE FREE', 'ACOFAR', 'ADDAX', 'AKILDIA', 'ALBOLENE', 'AMLACTIN', 'ANSEBIC', 'AQUA SOAP', 'AQUA-SOAP',
#     'AVITIL', 'AZULENNE', 'BACCIDE', 'BEAUTY PLUS', 'BEDOOK', 'BEPANTHEN/BEPANTHOL', 'BETAGRANULOS', 'BIAFINE',
#     'BIOBLAS', 'BIOCLIN', 'BIOLIQ', 'BIOXCIN', 'BLUE LIZARD', 'BODYSOL', 'BONAVEN', 'BOROLINE', 'CERAMOL',
#     'CERTAIN DRI', 'CETOPIC', 'CHICCO', 'CICAMEL', 'COOPER', 'COTARYL', 'CRISTALIA', 'DECUBAL', 'DERMAC',
#     'DERMACTIVE', 'DERMADRATE', 'DERMAGE', 'DERMAKERI', 'DERMENA', 'DERMON', 'DERSUPRIL', 'DEUMAVAN',
#     'DOCTISSIMO PARAPHARMACIE', 'DR.LI', 'DR.LIDERMO', 'DRAYEX', 'DX2', 'E45', 'ELDOPAQUE', 'EMOLIENTA', 'EMOLIN',
#     'EMOLIUM', 'EPIMAX', 'EVASOL', 'FARMOQUIMICA', 'FILTROSOL', 'FLUOCIN', 'FREI OEL (BOUHON)', 'GALENCO', 'GIFRER',
#     'GILBERT', 'GOLD BOND', 'HAMILTON', 'HIDRAFIL', 'HIPOSOL', 'HYALIX', 'IDROVEL', 'IHADA', 'INFASIL',
#     'INTERAPOTHEK', 'IRALTONE', 'ITANIDERM', 'KAMILODERM', 'KETOXIN', 'KINERASE', 'KORA', 'LACTIBON',
#     'LACTO CALAMINE', 'LETI', 'LIFAR', 'LIPODERM', 'LOTRIMIN', 'MARQUE VERTE', 'MICRORET', 'MITOSYL', 'MODERM',
#     'MULTIDERMOL', 'MUSSVITAL', 'NEUTRA LICE', 'NEUTRAPHARM', 'NORDIN', 'NUMIS', 'NUMIS MED', 'NURAPHARM',
#     'NUTREM', 'NUTRISIL', 'OILATUM', 'OILLAN', 'OSMIN', 'OTC IBERICA', 'PANVEL DERMATIV', 'PARABOTICA',
#     'PHARMACTIV', 'PHARMASEPT', 'PHISOHEX', 'PROCICAR', 'REGENERUM', 'RESTIV', 'RESTIVOIL', 'REVALESKIN', 'ROCHE',
#     'ROGE CAVAILLES', 'ROYALCARE', 'RUGARD (SCHEFFLER)', 'SALILEX', 'SALLVE', 'SARNA', 'SAUGELLA', 'SEBORADIN',
#     'SHADE', 'SMOOTH-E', 'SOLAR FOAM', 'S-OLE', 'SPECTRABAN', 'STANHOME FAMILY EXPERT', 'STIEFEL', 'STIEPROX',
#     'STIPROX', 'STIPROXAL', 'TARMED', 'TRACTOPON', 'TRI DERMA MD', 'UREADERM', 'UVEIL-PS', 'UVESOL', 'VEA',
#     'VENUSIA', 'VITA CITRAL', 'VITALIFE', 'ZODIAC','QV', 'BOBAI',
#     'COLLAGE','DERMAREST','EPIZONE E','GLAMY LAB','LU MILD','NOLAVER','OXECURE','RIUP','SEBCUR','SELENGENA','SEROPIPE','SHAAN','STAR VILLE','STRONGVILLE','SYNOBAR','UREMOL',
#     'URISEC','ZINPLEX'
# ]

mass_medic = [
    'ACNE AID', 'ACNES', 'AVEENO', 'BALNEUM', 'BENZAC', 'BIO-OIL', 'CARMEX', 'CERAVE', 'CETAPHIL', 'CUREL',
    'DERMATIX', 'DERMAVEEN', 'DIFFERIN','DR.YU', 'EGO', 'EUBOS', 'LACTACYD', 'LINOLA', 'MUSTELA',
    'NEUTROGENA', 'PANOXYL', 'PHYSIOGEL', 'SEBAMED', 'TOPICREM', 'VANICREAM', 'WIS', 'XHEKPON', 'FIRST AID BEAUTY',
    'AQUAPHOR', 'NOBACTER', 'LUBRIDERM', 'NEOSPORIN', 'DARROW', 'DEXERYL', 'ALERGIBON', 'ALPHYGIENE', 'BABIGOZ',
    'CANDERMYL', 'GALDERMA', 'GALDERMA OTHER', 'HELIOBLOC', 'HYDRODERM OMEGA', 'IOCON', 'IONIL', 'MACROLANE',
    'MICROBAN', 'MICROSUN', 'NESTLE', 'NUTRASPA', 'OBSERVANCE', 'PHYGIENE', 'R-GEN', 'SENTIAL', 'ACHE', 'ACNAID',
    'ACNE FREE', 'ACOFAR', 'ADDAX', 'AKILDIA', 'ALBOLENE', 'AMLACTIN', 'ANSEBIC', 'AQUA SOAP', 'AQUA-SOAP',
    'AVITIL', 'AZULENNE', 'BACCIDE', 'BEAUTY PLUS', 'BEDOOK', 'BEPANTHEN/BEPANTHOL', 'BETAGRANULOS', 'BIAFINE',
    'BIOBLAS', 'BIOCLIN', 'BIOLIQ', 'BIOXCIN', 'BLUE LIZARD', 'BODYSOL', 'BONAVEN', 'BOROLINE', 'CERAMOL',
    'CERTAIN DRI', 'CETOPIC', 'CHICCO', 'CICAMEL', 'COOPER', 'COTARYL', 'CRISTALIA', 'DECUBAL', 'DERMAC',
    'DERMACTIVE', 'DERMADRATE', 'DERMAGE', 'DERMAKERI', 'DERMENA', 'DERMON', 'DERSUPRIL', 'DEUMAVAN',
    'DOCTISSIMO PARAPHARMACIE', 'DR.LI', 'DR.LIDERMO', 'DRAYEX', 'DX2', 'E45', 'ELDOPAQUE', 'EMOLIENTA', 'EMOLIN',
    'EMOLIUM', 'EPIMAX', 'EVASOL', 'FARMOQUIMICA', 'FILTROSOL', 'FLUOCIN', 'FREI OEL (BOUHON)', 'GALENCO', 'GIFRER',
    'GILBERT', 'GOLD BOND', 'HAMILTON', 'HIDRAFIL', 'HIPOSOL', 'HYALIX', 'IDROVEL', 'IHADA', 'INFASIL',
    'INTERAPOTHEK', 'IRALTONE', 'ITANIDERM', 'KAMILODERM', 'KETOXIN', 'KINERASE', 'KORA', 'LACTIBON',
    'LACTO CALAMINE', 'LETI', 'LIFAR', 'LIPODERM', 'LOTRIMIN', 'MARQUE VERTE', 'MICRORET', 'MITOSYL', 'MODERM',
    'MULTIDERMOL', 'MUSSVITAL', 'NEUTRA LICE', 'NEUTRAPHARM', 'NORDIN', 'NUMIS', 'NUMIS MED', 'NURAPHARM',
    'NUTREM', 'NUTRISIL', 'OILATUM', 'OILLAN', 'OSMIN', 'OTC IBERICA', 'PANVEL DERMATIV', 'PARABOTICA',
    'PHARMACTIV', 'PHARMASEPT', 'PHISOHEX', 'PROCICAR', 'REGENERUM', 'RESTIV', 'RESTIVOIL', 'REVALESKIN', 'ROCHE',
    'ROGE CAVAILLES', 'ROYALCARE', 'RUGARD (SCHEFFLER)', 'SALILEX', 'SALLVE', 'SARNA', 'SAUGELLA', 'SEBORADIN',
    'SHADE', 'SMOOTH-E', 'SOLAR FOAM', 'S-OLE', 'SPECTRABAN', 'STANHOME FAMILY EXPERT', 'STIEFEL', 'STIEPROX',
    'STIPROX', 'STIPROXAL', 'TARMED', 'TRACTOPON', 'TRI DERMA MD', 'UREADERM', 'UVEIL-PS', 'UVESOL', 'VEA',
    'VENUSIA', 'VITA CITRAL', 'VITALIFE', 'ZODIAC','QV', 'BOBAI',
    'COLLAGE','DERMAREST','EPIZONE E','GLAMY LAB','LU MILD','NOLAVER','OXECURE','RIUP','SEBCUR','SELENGENA','SEROPIPE','SHAAN','STAR VILLE','STRONGVILLE','SYNOBAR','UREMOL',
    'URISEC','ZINPLEX'
]
print('Total mass_medic Brands: ', len(mass_medic))

Total mass_medic Brands:  216


In [8]:
# Group Brands by Loreal brand, Mass Medic and Market
def map_brand(row):
    if row['BRAND'] in ['GARNIER', 'MAYBELLINE','3CE'] :
        return row['BRAND']
    elif row['BRAND'] == 'L\'OREAL PARIS':
        return 'LOREAL PARIS'
    elif row['BRAND'] in  mass_medic:
        return 'Mass Medic'
    elif pd.isna(row['BRAND']):
        return 'Market'
    else:
        return 'Others'

In [9]:
def map_category(row):
    if row['GENDER'] == 'MEN' :
        return 'Male Skincare'
    elif row['GENDER'] == 'WOMAN' :
        return 'Female Skincare'
    else:
        return 'Others'

In [10]:
# Change Sales Value for Platform Nielsen to 0 - Total Market (only need value for Makeup)
def map_value(row):
    if row['Platform'] == 'Nielsen' :
        return 0
    else:
        return row['Sales Value']

In [11]:
# Important!! Try to understand the filter and logic here
for df in dfs:
    dfs[df]['Year'] = dfs[df]['Periods'].str.extract('(\d+)', expand=False).astype(int) + 2000
    dfs[df]['Month Name'] = dfs[df]['Periods'].str.split().str[0]
    dfs[df]['Month'] = dfs[df]['Month Name'].apply(month_to_number)
    dfs[df]['Platform'] = dfs[df].apply(map_platform, axis=1)
    dfs[df]['Brand'] = dfs[df].apply(map_brand, axis=1)

In [12]:
# Unique transformation for each category
dfs['MY Nielsen Skincare']['Category'] = dfs['MY Nielsen Skincare'].apply(map_category, axis=1)
dfs['MY Nielsen Skincare']['Sales Value'] = dfs['MY Nielsen Skincare'].apply(map_value, axis=1)
dfs['MY Nielsen Cosmetic']['Category'] = 'Makeup'
dfs['MY Nielsen Mass Medic']['Category'] = dfs['MY Nielsen Mass Medic'].apply(map_category, axis=1)
dfs['MY Nielsen Skincare'][dfs['MY Nielsen Skincare']['Brand'] == 'Market'].groupby('Year')['Sales Value'].sum().apply(lambda x: f"{x:,.0f}")

Year
2023      994,012,671
2024    1,463,941,495
2025    1,476,603,694
2026      629,165,833
Name: Sales Value, dtype: str

In [13]:
# Aggregation for different nielsen offline reports
nielsen_offline_skincare = dfs['MY Nielsen Skincare'].groupby(['Platform', 'Year', 'Month', 'Brand', 'Category'])['Sales Value'].sum().reset_index()
nielsen_offline_cosmetic = dfs['MY Nielsen Cosmetic'].groupby(['Platform', 'Year', 'Month', 'Brand', 'Category'])['Sales Value'].sum().reset_index()
nielsen_offline_mass_medic = dfs['MY Nielsen Mass Medic'][dfs['MY Nielsen Mass Medic']['Brand'] == 'Mass Medic'].groupby(['Platform', 'Year', 'Month', 'Brand', 'Category'])['Sales Value'].sum().reset_index()
dfs['MY Nielsen Skincare'][dfs['MY Nielsen Skincare']['Brand'] == 'Market'].groupby('Year')['Sales Value'].sum().apply(lambda x: f"{x:,.0f}")


Year
2023      994,012,671
2024    1,463,941,495
2025    1,476,603,694
2026      629,165,833
Name: Sales Value, dtype: str

In [14]:
# Merge nielsen offline reports
nielsen_my_cpd = pd.concat([nielsen_offline_skincare, nielsen_offline_cosmetic, nielsen_offline_mass_medic], ignore_index=True)
nielsen_my_cpd.tail()

,Platform,Year,Month,Brand,Category,Sales Value
2003,Nielsen,2026,1,Mass Medic,Female Skincare,5027712.344
2004,Nielsen,2026,2,Mass Medic,Female Skincare,4722781.319
2005,Nielsen,2026,3,Mass Medic,Female Skincare,5473642.349
2006,Nielsen,2026,4,Mass Medic,Female Skincare,4798126.474
2007,Nielsen,2026,5,Mass Medic,Female Skincare,5110686.379


### OMT CPD

In [15]:
# Important!! Make sure the file exist and updated first
files = glob.glob(f'{dir}/../Data Source/OMT - O+O/MY CPD/*.xlsx')

# check the list of files found
print("Files found:", files)

# List to store DataFrames
dfs = []

# Read each file and append to the list
for file in files:
    print(f"Reading file: {file}")
    df = pd.read_excel(file)
    dfs.append(df)
    print(f"File {file} read successfully")


# Concatenate all DataFrames into one
online_data = pd.concat(dfs, ignore_index=True)


Files found: ['c:\\Users\\balatarsini_avinitya\\Downloads\\CPD & LDB O+O - May\\loreal-report-automation (2)\\O+O/../Data Source/OMT - O+O/MY CPD\\OMT MY CPD 2024-01.xlsx', 'c:\\Users\\balatarsini_avinitya\\Downloads\\CPD & LDB O+O - May\\loreal-report-automation (2)\\O+O/../Data Source/OMT - O+O/MY CPD\\OMT MY CPD 2024-02.xlsx', 'c:\\Users\\balatarsini_avinitya\\Downloads\\CPD & LDB O+O - May\\loreal-report-automation (2)\\O+O/../Data Source/OMT - O+O/MY CPD\\OMT MY CPD 2024-03.xlsx', 'c:\\Users\\balatarsini_avinitya\\Downloads\\CPD & LDB O+O - May\\loreal-report-automation (2)\\O+O/../Data Source/OMT - O+O/MY CPD\\OMT MY CPD 2024-04.xlsx', 'c:\\Users\\balatarsini_avinitya\\Downloads\\CPD & LDB O+O - May\\loreal-report-automation (2)\\O+O/../Data Source/OMT - O+O/MY CPD\\OMT MY CPD 2024-05.xlsx', 'c:\\Users\\balatarsini_avinitya\\Downloads\\CPD & LDB O+O - May\\loreal-report-automation (2)\\O+O/../Data Source/OMT - O+O/MY CPD\\OMT MY CPD 2024-06.xlsx', 'c:\\Users\\balatarsini_avinitya

In [16]:
mapping = pd.read_excel(f'{dir}/../Data Source/CPD Skincare Mapping/Skincare Mapping.xlsx', sheet_name='MY')
# Catgory mapping
# def map_category(row):
#     if (row['Category L1'] == 'MAKEUP') & (row['Category L2_x'] in ['EYE MAKEUP','FACE MAKEUP','LIP MAKEUP', 'NAIL MAKEUP', 'OTHER MAKEUP']):
#         return 'Makeup'
#     elif (row['Category L1'] == 'SKIN CARE') & (row['Category L2_x'] in ['FACE CARE & CLEANSING', 'OTHER SKINCARE', 'SETS & PACKAGES']):
#         return 'Skincare'
#     elif (row['Category L1'] == 'SKIN CARE') & (row['Category L2_x'] in ['SUN CARE']):
#         return 'Suncare'
#     elif (row['Category L1'] == 'HAIR') & (row['Category L2_x'] == 'HAIR COLOR'):
#         return 'Hair Colour'
#     elif (row['Category L1'] == 'HAIR') & (row['Category L2_x'] in ['HAIR CARE','OTHER HAIR']):
#         return 'Hair Care'
#     else:
#         return 'Others'
    
# Catgory mapping
def map_category(row):
    if (row['Category L1'] == 'MAKEUP') & (row['Category L2_x'] in ['EYE MAKEUP','FACE MAKEUP','LIP MAKEUP', 'NAIL MAKEUP', 'OTHER MAKEUP']):
        return 'Makeup'
    elif (row['Category L1'] == 'SKIN CARE') & (row['Category L2_x'] in ['FACE CARE & CLEANSING']):
        return 'Skincare'
    elif (row['Category L1'] == 'SKIN CARE') & (row['Category L2_x'] in ['SUN CARE']) & (row['Category L3'] in ['FACE PROTECTION']):
        return 'Suncare'
    elif (row['Category L1'] == 'HAIR') & (row['Category L2_x'] == 'HAIR COLOR'):
        return 'Hair Colour'
    elif (row['Category L1'] == 'HAIR') & (row['Category L2_x'] in ['HAIR CARE']):
        return 'Hair Care'
    else:
        return 'Others'

In [17]:
# Rename Brand
def map_brand(row):
    if row['Brand_x'] == "L'OREAL PARIS" :
        return 'LOREAL PARIS'
    else:
        return row['Brand_x']

In [18]:
# Renaming columns for duplicated columns
online_data = online_data.rename(columns={'Mall Type': 'Platform', 'Total Est. Sales Local': 'Sales Value', 'Brand': 'Brand_x', 'Category L2': 'Category L2_x'})


In [19]:
# Important!! Try to understand the filter and logic here
online_data = online_data[online_data['Platform'].isin(['Shopee Mall', 'Lazada Mall','Tiktok Mall'])]
online_data = online_data[online_data['Category L1'] != 'FRAGRANCE']
online_data['Brand_x'] = online_data.apply(map_brand, axis=1)
online_data[['Year', 'Month']] = online_data['Year Month'].str.split('-', expand=True)
online_data['Brand'] = 'Market'
online_data['Category'] = online_data.apply(map_category, axis=1)
online_data

,Country,Year Month,Universe,Brand_x,Category L1,Category L2_x,Platform,Category L3,Product,Benefits,Formats,Sales Value,Loreal 1P Est Sales Local,Total units sold,Year,Month,Brand,Category
0,MY,2024-01,MASS,WM,SKIN CARE,FACE CARE & CLEANSING,Tiktok Mall,FACE MASK & PACKS,100pcs WM EXTRA COLLAGEN FIRMING SLEEPING MASK,ANTI AGING,OTHER FORMAT,1480900.59,NaN,64359.0,2024,01,Market,Skincare
1,MY,2024-01,MASS,BARTECH BEAUTEE,SKIN CARE,FACE CARE & CLEANSING,Tiktok Mall,FACIAL MOISTURIZER,20gm Krim Putih Bartech Beautee,BRIGHTENING,OTHER FORMAT,1265433.84,NaN,38104.0,2024,01,Market,Skincare
2,MY,2024-01,MASS,ANAS,SKIN CARE,FACE CARE & CLEANSING,Tiktok Mall,LIP TREATMENT,ANAS Eyeshadow Palette & ANAS Cream Blusher Set!,OTHER BENEFIT,CREAM,1096778.10,NaN,53790.0,2024,01,Market,Skincare
3,MY,2024-01,MASS,ANAS,SKIN CARE,FACE CARE & CLEANSING,Tiktok Mall,LIP TREATMENT,ANAS Glossy Glow Melting Lip Balm (9ML) dengan...,HYDRATING,BALM,848267.36,NaN,29291.0,2024,01,Market,Skincare
4,MY,2024-01,MASS,AXIS-Y,SKIN CARE,FACE CARE & CLEANSING,Tiktok Mall,SERUM & ESSENCE,AXIS-Y Dark Spot Correcting Glow Serum Best Se...,PIGMENTATION,SERUM,794601.20,NaN,8824.0,2024,01,Market,Skincare
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2005036,MY,2026-05,MASS,ROREC,SKIN CARE,FACE CARE & CLEANSING,Tiktok Mall,FACE MASK & PACKS,ROREC SADOER Real Pineapple Cartoon Plant Frui...,BRIGHTENING,SHEET,0.39,NaN,1.0,2026,05,Market,Skincare
2005037,MY,2026-05,MASS,SADOER,SKIN CARE,FACE CARE & CLEANSING,Tiktok Mall,FACE MASK & PACKS,[ROREC] SADOER 70% Rice Essence Whitening Nour...,ANTI AGING,SHEET,0.39,NaN,1.0,2026,05,Market,Skincare
2005038,MY,2026-05,MASS,SADOER,SKIN CARE,FACE CARE & CLEANSING,Tiktok Mall,FACE MASK & PACKS,[ROREC] SADOER Rosehip Hydrate Antioxidant Fac...,BRIGHTENING,SHEET,0.39,NaN,1.0,2026,05,Market,Skincare
2005039,MY,2026-05,MASS,ROREC,SKIN CARE,FACE CARE & CLEANSING,Lazada Mall,FACE MASK & PACKS,ROREC Natural Blueberry Moisturiser Face Mask,HYDRATING,CREAM,0.38,NaN,1.0,2026,05,Market,Skincare


In [20]:
# Filter and aggregate online data for market and brand
online_market = online_data.groupby(['Platform', 'Year', 'Month', 'Brand', 'Category'])['Sales Value'].sum().reset_index()
loreal_brand = online_data[online_data['Brand_x'].isin(['GARNIER','MAYBELLINE','LOREAL PARIS','3CE'])].groupby(['Platform', 'Year', 'Month', 'Brand_x', 'Category'])['Sales Value'].sum().reset_index()
loreal_brand = loreal_brand.rename(columns={'Brand_x': 'Brand', 'Sales Value': 'Sales Value'})


In [21]:
# Merge skincare data for female and male percentage split
my_skincare_split = pd.concat([online_market[online_market['Category'] == 'Skincare'], loreal_brand[loreal_brand['Category'] == 'Skincare']], ignore_index=True)
my_skincare_split

,Platform,Year,Month,Brand,Category,Sales Value
0,Lazada Mall,2024,01,Market,Skincare,2981436.94
1,Lazada Mall,2024,02,Market,Skincare,3037063.33
2,Lazada Mall,2024,03,Market,Skincare,5528688.32
3,Lazada Mall,2024,04,Market,Skincare,3177609.80
4,Lazada Mall,2024,05,Market,Skincare,3931865.64
...,...,...,...,...,...,...
371,Tiktok Mall,2026,04,LOREAL PARIS,Skincare,334531.56
372,Tiktok Mall,2026,04,MAYBELLINE,Skincare,3760.82
373,Tiktok Mall,2026,05,GARNIER,Skincare,515923.10
374,Tiktok Mall,2026,05,LOREAL PARIS,Skincare,464533.72


In [22]:
# SQL Query for male female percentage split using mapping file
query = f"""
        WITH mapping_tx AS (
            SELECT 
                Brand,
                Year,
                Month,
                Shopee,
                Lazada,
                Tiktok,
                'Female Skincare' AS Category
            FROM mapping
        
            UNION
        
            SELECT
                Brand,
                Year,
                Month,
                1 - Shopee AS Shopee,
                1 - Lazada AS Lazada,
                1 - Tiktok AS Tiktok,
                'Male Skincare' AS Category
            FROM mapping
        )

        SELECT
            a.Platform,
            a.Year,
            a.Month,
            a.Brand,
            b.Category,
            CASE
                WHEN a.Platform = 'Shopee Mall' THEN a."Sales Value" * b.Shopee
                WHEN a.Platform = 'Lazada Mall' THEN a."Sales Value" * b.Lazada
                WHEN a.Platform = 'Tiktok Mall' THEN a."Sales Value" * b.Tiktok
            END AS "Sales Value"
        FROM my_skincare_split a
            LEFT JOIN mapping_tx b
                ON (
                    a.Brand = b.Brand
                    AND a.Year = b.Year
                    AND a.Month = b.Month
                )
        """

my_skincare = sqldf(query)
my_skincare

,Platform,Year,Month,Brand,Category,Sales Value
0,Lazada Mall,2024,01,Market,Male Skincare,8.514479e+04
1,Lazada Mall,2024,01,Market,Female Skincare,2.896292e+06
2,Lazada Mall,2024,02,Market,Male Skincare,8.673339e+04
3,Lazada Mall,2024,02,Market,Female Skincare,2.950330e+06
4,Lazada Mall,2024,03,Market,Male Skincare,1.578900e+05
...,...,...,...,...,...,...
747,Tiktok Mall,2026,05,GARNIER,Female Skincare,4.925997e+05
748,Tiktok Mall,2026,05,LOREAL PARIS,Male Skincare,1.029788e+04
749,Tiktok Mall,2026,05,LOREAL PARIS,Female Skincare,4.542358e+05
750,Tiktok Mall,2026,05,MAYBELLINE,Male Skincare,0.000000e+00


In [23]:
# Exclude skincare from market and brand
online_market = online_market[~(online_market['Category'].isin(['Skincare', 'Others']))]
loreal_brand = loreal_brand[~(loreal_brand['Category'].isin(['Skincare', 'Others']))]

### OMT LDB DATA

In [24]:
# Important!! Make sure the file exist and updated first
dir = os.getcwd()
files = glob.glob(f'{dir}/../Data Source/OMT - O+O/MY LDB/*.xlsx')

# List to store DataFrames
dfs = []
# Read each file and append to the list
for file in files:
    df = pd.read_excel(file)
    dfs.append(df)

# Concatenate all DataFrames into one
ldb_data = pd.concat(dfs, ignore_index=True)

In [25]:
# Rename Platform
def map_platform(row):
    if row['Mall Type'] == 'Shopee Mall' :
        return 'Shopee Mall'
    elif row['Mall Type'] == 'Lazada Mall' :
        return 'Lazada Mall'
    elif row['Mall Type'] == 'Tiktok Mall' :
        return 'Tiktok Mall'
    else:
        return 'Others'

In [26]:
# Renaming columns for duplicated columns
ldb_data = ldb_data.rename(columns={'Total Est. Sales Local': 'Sales Value', 'Brand': 'Brand_x', 'Category L2': 'Category L2_x'})
print(ldb_data.columns[ldb_data.columns.duplicated()])
print(ldb_data.columns.tolist())



Index([], dtype='str')
['Country', 'Year Month', 'Universe', 'Brand_x', 'Category L1', 'Category L2_x', 'Mall Type', 'Category L3', 'Product', 'Benefits', 'Formats', 'Sales Value', 'Loreal 1P Est Sales Local', 'Total units sold']


In [27]:
# Important!! Try to understand the filter and logic here
ldb_data = ldb_data[ldb_data['Brand_x'].isin(mass_medic)]
ldb_data = ldb_data[ldb_data['Category L1'] == 'SKIN CARE']
ldb_data = ldb_data[ldb_data['Mall Type'].isin(['Shopee Mall', 'Lazada Mall','Tiktok Mall'])]
# ldb_data = ldb_data[~ldb_data['Category L2_x'].isin(['BODY CARE','SUN CARE'])]
ldb_data = ldb_data[
    ~(
        (ldb_data['Category L2_x'] == 'BODY CARE') |
        (
            (ldb_data['Category L2_x'] == 'SUN CARE') &
            (ldb_data['Category L3'] == 'FACE PROTECTION')
        )
    )
]
ldb_data['Platform'] = ldb_data.apply(map_platform, axis=1)
ldb_data[['Year', 'Month']] = ldb_data['Year Month'].str.split('-', expand=True)
ldb_data['Brand'] = 'Mass Medic'
ldb_data['Category'] = 'Female Skincare'
ldb_data.head()

,Country,Year Month,Universe,Brand_x,Category L1,Category L2_x,Mall Type,Category L3,Product,Benefits,Formats,Sales Value,Loreal 1P Est Sales Local,Total units sold,Platform,Year,Month,Brand,Category
21,MY,2024-01,MASS MEDICAL,NEUTROGENA,SKIN CARE,FACE CARE & CLEANSING,Shopee Mall,FACIAL MOISTURIZER,Neutrogena Hydro Boost Hyaluronic Acid Water G...,HYDRATING,GEL,33670.40,NaN,744.0,Shopee Mall,2024,01,Mass Medic,Female Skincare
49,MY,2024-01,MASS MEDICAL,CETAPHIL,SKIN CARE,FACE CARE & CLEANSING,Lazada Mall,FACIAL CLEANSER,CETAPHIL TWIN PACK GENTLE SKIN CLEANSER FOR FA...,OIL CONTROL,FACIAL CLEANSER,21468.00,NaN,120.0,Lazada Mall,2024,01,Mass Medic,Female Skincare
51,MY,2024-01,MASS MEDICAL,CETAPHIL,SKIN CARE,FACE CARE & CLEANSING,Shopee Mall,FACIAL CLEANSER,Cetaphil Gentle Skin Cleanser (500ml x 2),HYDRATING,FACIAL CLEANSER,21154.07,NaN,233.0,Shopee Mall,2024,01,Mass Medic,Female Skincare
62,MY,2024-01,MASS MEDICAL,CETAPHIL,SKIN CARE,FACE CARE & CLEANSING,Lazada Mall,FACIAL CLEANSER,CETAPHIL TWIN PACK GENTLE SKIN CLEANSER FOR FA...,OIL CONTROL,FACIAL CLEANSER,17188.26,NaN,186.0,Lazada Mall,2024,01,Mass Medic,Female Skincare
67,MY,2024-01,MASS MEDICAL,CETAPHIL,SKIN CARE,FACE CARE & CLEANSING,Shopee Mall,FACIAL CLEANSER,Cetaphil Gentle Skin Cleanser for Face & Body ...,HYDRATING,FACIAL CLEANSER,16078.86,NaN,126.0,Shopee Mall,2024,01,Mass Medic,Female Skincare


In [28]:
# Aggregate LDB data
ldb_market = ldb_data.groupby(['Platform', 'Year', 'Month', 'Brand', 'Category'])['Sales Value'].sum().reset_index()
ldb_market.head()

,Platform,Year,Month,Brand,Category,Sales Value
0,Lazada Mall,2024,01,Mass Medic,Female Skincare,250049.55
1,Lazada Mall,2024,02,Mass Medic,Female Skincare,312664.58
2,Lazada Mall,2024,03,Mass Medic,Female Skincare,510808.20
3,Lazada Mall,2024,04,Mass Medic,Female Skincare,216024.53
4,Lazada Mall,2024,05,Mass Medic,Female Skincare,306284.58


### Final Transformation

In [29]:
output = nielsen_my_cpd[nielsen_my_cpd.Brand == 'Market'].groupby(['Year','Category'])['Sales Value'].sum()
# Format with commas and no scientific notation
formatted_output = output.apply(lambda x: f"{x:,.0f}")

print(formatted_output)

Year  Category       
2023  Female Skincare      913,813,741
      Makeup               508,274,824
      Male Skincare         80,129,843
      Others                    69,087
2024  Female Skincare    1,354,984,350
      Makeup               707,663,730
      Male Skincare        108,495,008
      Others                   462,137
2025  Female Skincare    1,370,411,822
      Makeup               771,175,464
      Male Skincare        105,549,305
      Others                   642,567
2026  Female Skincare      582,541,372
      Makeup               350,809,882
      Male Skincare         46,282,999
      Others                   341,462
Name: Sales Value, dtype: str


In [30]:
# oo_my_cpd = pd.concat([nielsen_my_cpd], ignore_index=True)
oo_my_cpd = pd.concat([nielsen_my_cpd, online_market, loreal_brand, ldb_market, my_skincare], ignore_index=True)

# Remove Others category
oo_my_cpd = oo_my_cpd[oo_my_cpd['Category'] != 'Others']

# oo_my_cpd = pd.concat([nielsen_my_cpd], ignore_index=True)
oo_my_cpd = oo_my_cpd.sort_values(by=['Platform', 'Year', 'Month', 'Brand', 'Category'])
oo_my_cpd ['Month'] = oo_my_cpd ['Month'].astype(int)

In [31]:
oo_my_cpd['Year'] = oo_my_cpd['Year'].astype(int)
oo_my_cpd = oo_my_cpd[oo_my_cpd['Year'] >= 2021]
oo_my_cpd['Month'] = oo_my_cpd['Month'].astype(int)

In [32]:
# Get last month
last_month = datetime.now().replace(day=1) - timedelta(days=1)
month_abbr = last_month.strftime("%b").upper()

year = last_month.year

# Format the output as "MMM YYYY"
filemonth = last_month.strftime("%b %Y").upper()

# Print the result
print(f"{filemonth}")

MAY 2026


In [33]:
if not os.path.exists(f'../Generated Data/O+O/{filemonth}'):
        os.makedirs(f'../Generated Data/O+O/{filemonth}')

In [34]:
oo_my_cpd.to_excel(f'../Generated Data/O+O/{filemonth}/MY CPD {filemonth} O+O.xlsx', index=False)